# Unified Kaggle Gate 2

You can use a NEW private notebook with GPU T4 x2 and Internet enabled. Attach the new private dataset containing `unified_gate2_20260928_v5.bin`. No remote API key or tunnel is needed. HF_TOKEN is optional if model download requires your account.

You can run the three code cells in order, then download evidence.zip and return Cell 3 output. The experiment uses GPU 0 only. Keep the session running until Cell 2 exits. Gate 2 requires independent evidence review. Existing runtimes and Gate 1 evidence remain unchanged. Gate 3 is not included.


In [ ]:
from pathlib import Path
from hashlib import sha256
import json
import zipfile

BUNDLE_NAME = "unified_gate2_20260928_v5.bin"
BUNDLE_SHA256 = "0e5ffd854f394f86e5c1ca49d47b9b230b0088aae2da828d893601330d190feb"
ROOT = Path("/kaggle/working/gate2_bundle")
OUT = Path("/kaggle/working/unified_gate2")

matches = list(Path("/kaggle/input").rglob(BUNDLE_NAME))
if len(matches) != 1:
    raise RuntimeError("Attach exactly one private Gate 2 bundle dataset.")
if ROOT.exists() or OUT.exists():
    raise RuntimeError("Use a fresh notebook session. Existing experiment files will not be overwritten.")
digest = sha256()
with matches[0].open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
if digest.hexdigest() != BUNDLE_SHA256:
    raise RuntimeError("Wrong or corrupt Gate 2 upload. Do not extract it.")
with zipfile.ZipFile(matches[0]) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)) or archive.testzip() is not None:
        raise RuntimeError("Bundle inventory or CRC failed.")
    manifest = json.loads(archive.read("gate2_bundle.json"))
    if manifest.get("schema") != "unified-kaggle-gate2-v1":
        raise RuntimeError("Not a Gate 2 bundle.")
    if set(names) != set(manifest["files"]) | {"gate2_bundle.json"}:
        raise RuntimeError("Unexpected bundle member.")
    for name in names:
        path = Path(name)
        target = (ROOT / path).resolve()
        if path.is_absolute() or ".." in path.parts or "\\" in name or ":" in name or not target.is_relative_to(ROOT.resolve()):
            raise RuntimeError("Unsafe bundle path.")
        if name in manifest["files"] and sha256(archive.read(name)).hexdigest() != manifest["files"][name]:
            raise RuntimeError("Bundle file hash failed.")
    ROOT.mkdir(parents=True)
    archive.extractall(ROOT)
print("GATE_2_BUNDLE_VERIFIED", len(names), "members")
print("Next: run Cell 2. No GPU inference has run yet.")


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(ROOT / "scripts/unified_gate2_bootstrap.py"), "--output", str(OUT)],
    cwd=ROOT,
)
print("Bootstrap exit code:", result.returncode)
print("Run Cell 3 even if setup or switching failed, so partial evidence can be downloaded.")


In [ ]:
from IPython.display import FileLink, display

setup_path = OUT / "setup.json"
summary_path = OUT / "results/summary.json"
if setup_path.exists():
    setup = json.loads(setup_path.read_text())
    print("Setup status:", setup["status"], "phase:", setup["phase"])
    if setup.get("error"):
        print("Recorded failure:", setup["error"])
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("Probe status:", summary["status"])
    print("Foundation load count:", summary["foundation_load_count"])
    for row in summary["results"]:
        print(row["label"], row["requested_feature"], row["requested_style"],
              "Gate 1 RGB equality:", row["versus_gate1"]["rgb_pixels_equal"],
              "restored RGB equality:", (row["versus_first_shared"] or {}).get("rgb_pixels_equal"),
              "adapter:", row["active_adapter_id"])
    print("Recovery/concurrency probes:", summary["recovery"])
    print("Isolated repeat features:", summary.get("isolated_repeat_required_features", []))
print("No automatic Gate 2 pass. Gate 3 has not started.")
if (OUT / "evidence.zip").exists():
    display(FileLink(str(OUT / "evidence.zip")))
else:
    print("No evidence archive exists yet. Return the cell output if bootstrap never started.")
